# VectorDrugMatch - final Colab run (PrimeKG, both benchmark variants)

Run the cells **top to bottom**. Everything is driven by `scripts/run_full_pipeline.py`; the code is **pinned to commit `cb41c4406e`** so the run is reproducible.
Before running: that commit must be pushed to GitHub (`origin/mukul`), and your private files must be in Drive (Step 3).
Training happens only here; local work produced the data layer, audits and exporters. Read `Docs/DATA_AND_PIPELINE_READINESS.md` first.

## Step 1 - GPU
Runtime > Change runtime type > T4 GPU. Expect `GPU available: True`.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi | head -15

## Step 2 - Drive, code (pinned commit) and libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/VDM_Data /content/drive/MyDrive/VDM_Outputs
%cd /content
!rm -rf VectorDrugMatch
!git clone https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
PIN = 'cb41c4406e0d8663f64bbea0af3d159ead20ca1e'
!git checkout $PIN
!git log -1 --oneline

In [ ]:
!pip install -q rdkit mlflow optuna shap xgboost lightgbm loguru pyyaml torch-geometric tabulate scikit-learn lxml matplotlib pymupdf pyarrow
import torch
v = torch.__version__.split('+')[0]; c = (torch.version.cuda or '').replace('.', '')
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
import torch_geometric, torch_cluster
print('ok:', torch.__version__, torch_geometric.__version__, 'torch_cluster available')

In [ ]:
!mkdir -p outputs && pip freeze > outputs/pip_freeze_colab.txt && wc -l outputs/pip_freeze_colab.txt

## Step 3 - Put your private/manual inputs in place (from Drive)
Everything comes from `MyDrive/VDM_Data/` (see `Docs/FINAL_PROJECT_HANDOFF.md` section 6 for the exact layout):
`mondo.obo` (the frozen MONDO release; **required**, otherwise the latest MONDO is downloaded and the benchmark can drift), `drugbank_smiles_cache.json`, `drugbank_cid_map.json`,
`downloads/` (Open Targets, HAI, IHME), `B09474-eng.pdf` and optionally `who_eml.csv`, and `frozen_processed/` (safety copy of the frozen benchmark tables).
Missing optional items are fine: the matching equity components are reported as **missing**, never as zero.

In [ ]:
import os, shutil
D = '/content/drive/MyDrive/VDM_Data'
os.makedirs('data/external', exist_ok=True)
if os.path.isdir(f'{D}/downloads'):
    shutil.copytree(f'{D}/downloads', 'data/external/downloads', dirs_exist_ok=True); print('copied downloads/')
else:
    print('no downloads/ in Drive: Open Targets, HAI and IHME steps will be skipped (optional)')
for fn in ('mondo.obo', 'who_eml.csv', 'B09474-eng.pdf', 'drugbank_smiles_cache.json', 'drugbank_cid_map.json'):
    if os.path.exists(f'{D}/{fn}'):
        shutil.copy(f'{D}/{fn}', f'data/external/{fn}'); print('copied', fn)
    else:
        print('NOT in Drive:', fn)
assert os.path.exists('data/external/mondo.obo'), 'upload mondo.obo (frozen MONDO release 2026-11-03) to MyDrive/VDM_Data first'

## Step 4 - Prepare: download PrimeKG + MONDO, build tables, save split, run the safety check, build the structure-covered variant
Stops at the first failure. A validation `[FAIL]` means: do not train.

In [ ]:
!python scripts/run_full_pipeline.py --stage prepare

### Step 4b - Verify the benchmark is the frozen one (do not continue on a mismatch)
Compares SHA-256 of the regenerated tables, features and saved splits with `Docs/release/vdm_v1_candidate_manifest.json` (frozen locally). If a processed file differs, the frozen copy from
`MyDrive/VDM_Data/frozen_processed/` is restored and re-checked. A differing `mondo.obo` or `primekg_kg.csv` is fatal (fix the input, do not train).

In [ ]:
import hashlib, json, os, shutil
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
KEEP = ('.csv', '.npy', '.npz', '.obo')
want = {f['path']: f['sha256'] for f in man if f['path'].endswith(KEEP) and
        f['path'].startswith(('data/processed/primekg/', 'data/processed/primekg_structcov/', 'data/external/mondo.obo', 'data/external/primekg_kg.csv'))}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()
def check():
    return [p for p, h in want.items() if not os.path.exists(p) or sha(p) != h]
bad = check()
if bad:
    print('MISMATCH:', bad)
    fatal = [p for p in bad if p.startswith('data/external/')]
    assert not fatal, f'input differs from the frozen release: {fatal}'
    for name in ('primekg', 'primekg_structcov'):
        src = f'/content/drive/MyDrive/VDM_Data/frozen_processed/{name}'
        assert os.path.isdir(src), f'no frozen copy at {src}'
        shutil.copytree(src, f'data/processed/{name}', dirs_exist_ok=True)
    bad = check()
assert not bad, f'frozen benchmark NOT reproduced: {bad}'
print(f'FROZEN BENCHMARK VERIFIED: {len(want)} files match the release manifest')

## Step 5 - External evidence and the equity table (Open Targets, HAI, IHME, EML are optional inputs)

In [ ]:
!python scripts/run_full_pipeline.py --stage external

## Step 6 - Smoke test (do this before the long run)
Checks `torch_cluster`, Node2Vec for one epoch, five epochs for the five GNNs, one baseline. Look at runtime and GPU memory.

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python scripts/run_full_pipeline.py --stage smoke

## Step 7 - Full benchmark: baselines, Node2Vec, five GNNs, cold-start ablation, then evaluation and the paper results package
Baselines run one model per call. If Colab disconnects, re-run Steps 1-3 and this step: finished prediction files are rewritten, nothing else is affected.

In [ ]:
!python scripts/run_full_pipeline.py --stage train --variant full
!python scripts/run_full_pipeline.py --stage report --variant full
!mkdir -p /content/drive/MyDrive/VDM_Outputs && cp -r outputs/primekg outputs/primekg_dp /content/drive/MyDrive/VDM_Outputs/ 2>/dev/null || true

## Step 7b - Explanations on TEST pairs (SHAP for Random Forest / XGBoost, GNNExplainer for HGT and RGCN)
Only after Step 7. Explains the saved cold-start models, not the pooled all-data model. Attributions describe the model, not causal biology.

In [ ]:
!python scripts/run_full_pipeline.py --stage explain --variant full
!cp -r outputs/primekg/explain /content/drive/MyDrive/VDM_Outputs/primekg_full_explain 2>/dev/null || true

## Step 8 - Structure-covered variant (same drug partition, drugs with a usable fingerprint only)

In [ ]:
!python scripts/run_full_pipeline.py --stage train --variant structcov
!python scripts/run_full_pipeline.py --stage report --variant structcov
!cp -r outputs/primekg_structcov outputs/primekg_structcov_dp /content/drive/MyDrive/VDM_Outputs/ 2>/dev/null || true

## Step 9 - Freeze, package and download
Writes the release manifest (checksums, versions, commit), zips the results package, saves a copy to Drive and downloads it.

In [ ]:
!python scripts/freeze_release.py --tag vdm_v1
!python scripts/make_readiness_report.py --skip-validators
!zip -rq vdm_results.zip outputs/primekg/paper_package outputs/primekg_structcov/paper_package outputs/primekg/*.csv outputs/primekg_structcov/*.csv outputs/primekg_dp/*.csv outputs/primekg_structcov_dp/*.csv outputs/primekg/models/*.json outputs/primekg_structcov/models/*.json outputs/pipeline_log.jsonl outputs/pip_freeze_colab.txt outputs/primekg/explain Docs/release data/processed/equity data/processed/ntd data/processed/open_targets data/processed/hai
!cp vdm_results.zip /content/drive/MyDrive/VDM_Outputs/
from google.colab import files
files.download('vdm_results.zip')

## Reading the results honestly
Report the `all` rows first; show `with_structure` / `no_structure` next to them; NTD rows have ~15 test positives (exploratory). Baselines come in two variants (`primekg` = drug-only; `primekg_dp` = drug + train-side disease profile): report both, never pooled. Use only `coldstart_*` baseline numbers (the printed 5-fold CV pools drugs and is easier). The legacy `train_benchmark.py` Node2Vec and `run_ablations.py` are not drug-disjoint: never report them.